In [4]:
# this is a simple notebook demo for tokenization for cvtransformer
# note that for stable performance, centering and scaling should be used for numeric values.
# for a "concept" that has both numeric and non-numeric vlaues, two separate tokens should be used in the vocab
# glucose 140 -> ('glucose',140)
# glucose high -> ('glucose_high',nan)

import pandas as pd
import numpy as np

In [7]:
# read in the demodata ensuring time is read as time
df = pd.read_csv("./demodata.csv", parse_dates=['time'])

In [ ]:
# extremely simple tokenization scheme
# iter over rows
# if new id, add sos
# add time token if time changes

c = []
v = []

for i, row in df.iterrows():
    if i == 0 or row['id'] != df.iloc[i-1]['id']:
        c.append('<|sos|>')  # sos token
        v.append(np.nan)  # this is a non-numeric token
    # if time changes insert <|time|> token and delta time value
    if i > 0 and row['time'] != df.iloc[i-1]['time']:
        c.append('<|time|>')  # time token
        delta_t = row['time'] - df.iloc[i-1]['time']
        v.append(delta_t.total_seconds())
    # otherwise just append the data
    c.append(row['class'])
    v.append(float(row['value']))

In [19]:
# print parallel c,v with good justification of second col
for ci, vi in zip(c, v):
    print(f"{ci:<35} {vi}")

<|sos|>                             nan
ICD10:E11.9                         nan
ICD10:I10                           nan
<|time|>                            300.0
VITAL:Heart Rate                    82.0
VITAL:Systolic Blood Pressure       138.0
VITAL:Diastolic Blood Pressure      86.0
<|time|>                            1500.0
LAB:Hemoglobin A1c                  7.4
LAB:Creatinine                      1.1
LAB:Sodium                          139.0
<|time|>                            1500.0
MED:Metformin                       nan
MED:Lisinopril                      nan
<|sos|>                             nan
<|time|>                            611520.0
ICD10:J18.9                         nan
<|time|>                            180.0
VITAL:Heart Rate                    104.0
VITAL:Systolic Blood Pressure       112.0
VITAL:Respiratory Rate              22.0
<|time|>                            1500.0
LAB:White Blood Cell Count          14.2
LAB:Lactate                         2.3
<|time|>  

In [20]:
# fake tokenizer

class SimpleTokenizer:
    def __init__(self, concepts):
        self.concept_to_id = {concept: idx for idx, concept in enumerate(concepts)}
        self.id_to_concept = {idx: concept for concept, idx in self.concept_to_id.items()}

    def encode(self, concepts):
        return [self.concept_to_id[concept] for concept in concepts]

    def decode(self, ids):
        return [self.id_to_concept[idx] for idx in ids]

In [24]:
tok = SimpleTokenizer(concepts=set(c))
encoded_c = tok.encode(c)
decoded_c = tok.decode(encoded_c)

#print tokens, encoded, decoded, values
# header
print(f"{'Concept':<35} {'ID':<5} {'Decoded':<35} {'Value'}")
print("-"*85)
for ci, ei, di, vi in zip(c, encoded_c, decoded_c, v):
    print(f"{ci:<35} {ei:<5} {di:<35} {vi}")       

Concept                             ID    Decoded                             Value
-------------------------------------------------------------------------------------
<|sos|>                             0     <|sos|>                             nan
ICD10:E11.9                         11    ICD10:E11.9                         nan
ICD10:I10                           20    ICD10:I10                           nan
<|time|>                            13    <|time|>                            300.0
VITAL:Heart Rate                    7     VITAL:Heart Rate                    82.0
VITAL:Systolic Blood Pressure       14    VITAL:Systolic Blood Pressure       138.0
VITAL:Diastolic Blood Pressure      12    VITAL:Diastolic Blood Pressure      86.0
<|time|>                            13    <|time|>                            1500.0
LAB:Hemoglobin A1c                  18    LAB:Hemoglobin A1c                  7.4
LAB:Creatinine                      5     LAB:Creatinine                      1.1
L